# Building a Leak-Safe, Tuned Classification Pipeline

This project builds a full classification pipeline for predicting 
survival on the Titanic dataset, with a deliberate focus on the parts of 
a machine learning workflow that are easy to get wrong quietly: how 
missing data gets handled, how categorical features get encoded, whether 
preprocessing leaks information from test data into training, whether a 
single train/test split can be trusted at all, and which metric actually 
matters for the problem at hand. The goal wasn't just a model that 
predicts survival - it was a pipeline built to survive real evaluation, 
not just look good on one lucky split.

## Building It Leak-Safe

Every preprocessing step - imputing missing values, scaling numeric features,
one-hot encoding categorical ones - is fit only on training data and applied to
test data without refitting, packaged into a single scikit-learn Pipeline with
a ColumnTransformer routing each feature type to the right preprocessing steps.
This isn't a cosmetic choice: preprocessing steps fit on the full dataset before
splitting can silently leak test information into training, inflating reported 
performance in a way that would never hold up once the model sees genuinely unseen data.

## Evaluating It Honestly

Rather than relying on a single 80/20 split, the model was evaluated 
using 5-fold stratified cross-validation, which confirmed that a single 
split's reported accuracy can swing by several percentage points purely 
by chance - and, more importantly, revealed that recall varies far more 
across folds than accuracy does, since it's computed over a much smaller 
subset of the data (the minority Survived class specifically).

## Regularization Sweep

Logistic regression's `C` parameter controls how strongly coefficients 
are penalized - a smaller C means stronger regularization. Five values 
were tested with the full pipeline - [0.01, 0.1, 1, 10, 100] - each 
evaluated with 5-fold stratified cross-validation rather than a single 
split.

### Setup
imports, data, preprocessing, features

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


df = pd.read_csv('https://raw.githubusercontent.com/pandas-dev/pandas/master/doc/data/titanic.csv')

numeric_features = ['Age', 'Fare', 'SibSp', 'Parch']
categorical_features = ['Sex', 'Embarked']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
    ('pass', 'passthrough', ['Pclass'])
])

features = numeric_features + categorical_features + ['Pclass']
X = df[features]
y = df['Survived']

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

C=0.01: accuracy=0.7396, recall=0.3773, precision=0.8735, f1=0.5257
C=0.1: accuracy=0.7980, recall=0.6841, precision=0.7646, f1=0.7218
C=1: accuracy=0.7935, recall=0.7074, precision=0.7441, f1=0.7242
C=10: accuracy=0.7913, recall=0.7074, precision=0.7395, f1=0.7221
C=100: accuracy=0.7913, recall=0.7074, precision=0.7395, f1=0.7221


### Sweep

In [ ]:
C_values = [0.01, 0.1, 1, 10, 100]

for c in C_values:
    pipeline_c = Pipeline([
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(l1_ratio=0, C=c, max_iter=1000))
    ])
    
    results = cross_validate(pipeline_c, X, y, cv=skf, scoring=['accuracy', 'recall', 'precision', 'f1'])
    
    print(f"C={c}: accuracy={results['test_accuracy'].mean():.4f}, recall={results['test_recall'].mean():.4f}, "
          f"precision={results['test_precision'].mean():.4f}, f1={results['test_f1'].mean():.4f}")

## Sweep Results

| C | Accuracy | Recall | Precision | F1 |
|---|---|---|---|---|
| 0.01 | 0.7396 | 0.3773 | 0.8735 | 0.5257 |
| 0.1 | 0.7980 | 0.6841 | 0.7646 | 0.7218 |
| 1 | 0.7935 | 0.7074 | 0.7441 | 0.7242 |
| 10 | 0.7913 | 0.7074 | 0.7395 | 0.7221 |
| 100 | 0.7913 | 0.7074 | 0.7395 | 0.7221 |

Very strong regularization (C=0.01) makes the model overly cautious: 
precision is highest (0.87) but recall collapses to 0.38, since the model 
rarely predicts "Survived" at all. As C increases, this reverses - recall 
rises while precision steadily falls, the standard precision/recall 
tradeoff. From C=1 onward, both recall and precision essentially plateau, 
meaning the model isn't sensitive to weakening the regularization further 
- there's no real cost to more capacity here, but a real cost to over-
constraining it.

**C=1 was chosen** as the best balance: it achieves the same peak recall 
(0.7074) as C=10 and C=100, but with a slightly better F1 score (0.7242 
vs 0.7221) - making it the smallest, simplest model that reaches full 
performance, rather than an arbitrarily larger C with no added benefit.

## A Note on Evaluation Method

There's no separate train/test split anywhere in this project, and that's 
deliberate rather than an oversight. Every score reported above came from 
5-fold cross-validation - each fold takes a turn being held-out test data, 
the model is refit fresh each time, and the results are averaged. That's 
a more thorough version of what a single train/test split does, not a 
shortcut around it: choosing C=1 was based on comparing performance 
across 5 different held-out sets, not one, so the choice isn't resting on 
whatever happened to land in a single lucky or unlucky split.

The one exception is the final `.fit(X, y)` call used to inspect 
coefficients below - that step isn't evaluating anything, so training on 
the full dataset is the right choice there: more data gives a clearer, 
more stable picture of what the model actually learned, with no 
performance claim attached to that particular fit.

### The Final Model

In [ ]:
final_model = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(l1_ratio=0, C=1, max_iter=1000))
])

final_model.fit(X, y)

feature_names = final_model.named_steps['preprocessor'].get_feature_names_out()
coefficients = final_model.named_steps['model'].coef_[0]

coef_df = pd.DataFrame({'feature': feature_names, 'coefficient': coefficients})
coef_df['abs_coefficient'] = coef_df['coefficient'].abs()
print(coef_df.sort_values('abs_coefficient', ascending=False))

           feature  coefficient  abs_coefficient
4    cat__Sex_male    -2.617251         2.617251
7     pass__Pclass    -1.065086         1.065086
0         num__Age    -0.492783         0.492783
6  cat__Embarked_S    -0.385544         0.385544
2       num__SibSp    -0.344789         0.344789
1        num__Fare     0.107106         0.107106
3       num__Parch    -0.063768         0.063768
5  cat__Embarked_Q    -0.030711         0.030711


## Choosing the Final Model

C=1 was selected as covered above - matches the best recall while giving 
the strongest F1.

**Results:**

| Feature | Coefficient | 
|---|---|
| Sex_male | -2.617 |
| Pclass | -1.065 |
| Age | -0.493 |
| Embarked_S | -0.386 |
| SibSp | -0.345 |
| Fare | 0.107 |
| Parch | -0.064 |
| Embarked_Q | -0.031 |

Sex is by far the strongest predictor here - more than double the next 
closest feature, Pclass - meaning being male sharply lowers the model's 
predicted odds of survival, more than any other single factor in the 
dataset. Age, by comparison, has a fairly small coefficient relative to 
Sex and Pclass, meaning it plays a real but secondary role.

That coefficient size is also why a simpler imputation method was enough 
here: since Age isn't carrying much of the model's decision-making, 
there wasn't a strong case for a more complex imputation approach - the 
choice of how missing Age values got filled in wasn't going to move the 
model's behavior much either way.

## Takeaway

The final model's accuracy is not the interesting part of this project - the discipline behind it is. Every stage was built to guard against a specific way a model can quietly overstate its own quality: preprocessing steps that leak information from test data into training, a single lucky split that hides how much a score can swing, and a default metric (accuracy) that hides a much worse weakness underneath it (a model that misses 3 in 10 actual survivors).

The regularization sweep also made a subtler point worth stating plainly: pushing regularization too far doesn't just make a model "worse" evenly across the board - it distorts precision and recall in opposite directions, since a too-cautious model gets good at looking right without actually catching what matters. Choosing C by F1 rather than accuracy or recall alone is what surfaced that tradeoff clearly enough to make a real, defensible choice instead of an arbitrary one.

None of this changes the final number much. What it changes is how much that number can actually be trusted.